This OPTIONAL script takes an external QC spreadsheet (if you have one) and merges those QC judgments into your master data catalogue, filling per-session QC fields like “pass/fail” and optional notes. If QC is not available (or is disabled), it leaves the catalogue alone (aside from marking QC fields as unknown, depending on your settings).

It also computes simple derived summaries like “which scan is the first one that passed QC” and “how many scans passed QC,” which downstream steps can use to automatically pick which recordings to analyze.

**USAGE NOTES:**
- Will automatically skip QC decoration if the data catalogue contains any files with 'UNKNOWN' session_ID tags (this makes mapping too ambiguous)
- If using QC *and* employing "chronology-based" filtering preset options (e.g. 'first_good', 'last_good'), you must make sure that the 'session_ID_mappings' CONFIG field contains dictionary keys for ALL session_IDs that appear in the dataset, defined in chronological order. If any possible 'session_ID' labels are missing, the corresponding session_ID type will be *excluded* from chronological ordering (possibly "silently").


-------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path
import subprocess

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import yaml, os, json, re
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
# Set paths- / parameters:

ROOT_DIRECTORY = config['root_output_directory']
DATA_CATALOGUE = ROOT_DIRECTORY + 'master_data_catalogue.csv'
QC_TABLE_FILEPATH = config['QC_table_filepath']

HARD_STOP = config['hard_errors']

SKIP_CATALOGUE_QC = config['skip_catalogue_QC']

QC_COLUMN_MAP = config['QC_column_map']

PASS_QC_LABELS = config ['QC_pass_labels']
TIME_POINTS_DICT = config['session_ID_mappings']

# Prints:
print(f"--> Loading base data catalogue from: {DATA_CATALOGUE}")

if not SKIP_CATALOGUE_QC:
    print(f"--> Loading QC metadata table from: {QC_TABLE_FILEPATH}")

    # Print QC table column re-mappings (if applicable):
    counter = 0
    for key, value in QC_COLUMN_MAP.items():
        if pd.notna(value) and str(value) != str(key):
            counter += 1
            print(f"\t--> Re-mapping input column '{key}' → '{value}'")
    if counter == 0:
        print("\t--> No QC table column re-name mappings set; expecting default target column names.")
    else:
        print("\t--> Expecting default names for all other target columns")
else:
    print(f"\n[WARNING:] 'skip_catalogue_QC' metaparameter set to '{SKIP_CATALOGUE_QC}':\n   --> Skipping QC stage; base data catalogue will remain unchanged.")

In [ ]:
# Load data catalogue and QC metadata table:
data_df = pd.read_csv(DATA_CATALOGUE)

# ------------------------------------------------------------
# Preflight: skip QC stage if catalogue contains 'UNKNOWN' session_IDs (anywhere, e.g. MRI *or* MEG)
# (Ambiguous QC mapping == when input files lack BIDS 'ses-' labels)
# ------------------------------------------------------------
if not SKIP_CATALOGUE_QC:
    unknown_columns_detected = [c for c in data_df.columns if "_UNKNOWN" in str(c)]
    if unknown_columns_detected:
        print("[warn] Detected 'UNKNOWN' session-derived columns in master_data_catalogue.csv.")
        print("[warn] QC transfer is ambiguous for subjects with multiple MRI/MEG files lacking BIDS session ('ses-') labels.")
        print("[warn] Forcing SKIP_CATALOGUE_QC=True; leaving base catalogue unchanged.")
        print("[warn] Example columns:", ", ".join(sorted(unknown_columns_detected)[:10]))
        SKIP_CATALOGUE_QC = True

if not SKIP_CATALOGUE_QC:
    QC_df = pd.read_csv(QC_TABLE_FILEPATH)
    QC_df = QC_df.rename(columns={k: v for k, v in QC_COLUMN_MAP.items() if pd.notna(v)})

In [ ]:
# =========================
# Transfer QC fields from QC_df → data_df
# Fills: <data_type>_<session_ID>_QC_summary (required, error if missing)
#        <data_type>_<session_ID>_QC_comment (optional; write NaN if missing)
# =========================

if not SKIP_CATALOGUE_QC:

    # --- Required columns sanity checks ---
    for dataframe_name, dataframe_obj, required_columns in [
        ("QC_df",  QC_df,  ["subject_ID", "session_ID", "data_type", "QC_summary", "QC_comment"]),
        ("data_df", data_df, ["subject_ID"])]:
        if dataframe_name not in globals():
            raise NameError(f"{dataframe_name} not found.")
        missing_columns = [c for c in required_columns if c not in dataframe_obj.columns]
        if missing_columns:
            raise KeyError(f"{dataframe_name} missing required columns: {missing_columns}")

    # --- Optional: detect duplicates per (subject_ID, data_type, session_ID) in QC_df ---
    duplicate_key_counts = (
        QC_df.dropna(subset=["subject_ID", "data_type", "session_ID"])
            .groupby(["subject_ID", "data_type", "session_ID"])
            .size()
            .reset_index(name="row_count"))
    duplicate_rows = duplicate_key_counts[duplicate_key_counts["row_count"] > 1]
    if not duplicate_rows.empty:
        example_dups = duplicate_rows.head(10).to_dict(orient="records")
        raise ValueError(
            "[QC_df] Found duplicate rows for the same (subject_ID, data_type, session_ID). "
            f"Examples (first 10): {example_dups}")

    # --- Catalogue schema: valid (data_type, session_ID) pairs come from *_filename columns ---
    # This prevents QC_df from creating QC columns for sessions that do not exist in the catalogue.
    filename_pattern = re.compile(r"^(?P<dtype>[^_]+)_(?P<session>.+)_filename$")

    valid_dtype_session_pairs = set()
    for col in data_df.columns:
        match = filename_pattern.match(str(col))
        if not match:
            continue
        dtype = match.group("dtype").strip()
        session = match.group("session").strip()
        valid_dtype_session_pairs.add((dtype, session))

    if not valid_dtype_session_pairs:
        msg = "[QC_df] Could not infer any valid (data_type, session_ID) pairs from data_df '*_filename' columns. " \
              "QC transfer cannot be validated safely."
        if HARD_STOP:
            raise ValueError(msg)
        else:
            print("[warn]", msg)

    # --- Build a quick subject set for fast membership tests ---
    data_df_subjects = set(data_df["subject_ID"].dropna().astype(str))

    # Track columns we auto-create (so we don't spam warnings repeatedly)
    created_columns = set()
    updates_made = 0
    subjects_with_updates = set()

    # --- Main transfer loop ---
    for subject_identifier in QC_df["subject_ID"].dropna().astype(str).unique():
        if subject_identifier not in data_df_subjects:
            # No match in data_df — skip silently per your spec
            continue

        subject_rows_qc = QC_df[QC_df["subject_ID"].astype(str) == subject_identifier]

        for row_index in subject_rows_qc.index:
            data_type_label = subject_rows_qc.at[row_index, "data_type"]
            session_label = subject_rows_qc.at[row_index, "session_ID"]

            # Skip rows with missing data_type/session_ID (shouldn't happen, but be defensive)
            if pd.isna(data_type_label) or pd.isna(session_label):
                print(f"[warn] Skipping QC row with missing data_type/session_ID for subject_ID={subject_identifier}")
                continue

            data_type_label = str(data_type_label).strip()
            session_label = str(session_label).strip()

            # Validate QC row aligns to an existing catalogue slot (dtype/session)
            key = (data_type_label, session_label)
            if valid_dtype_session_pairs and (key not in valid_dtype_session_pairs):
                msg = (
                    "[QC_df] QC row does not match any existing catalogue (data_type, session_ID) slot: "
                    f"{key}. This would create QC columns that do not correspond to any file in master_data_catalogue.")
                if HARD_STOP:
                    raise ValueError(msg)
                else:
                    print("[warn]", msg)
                    continue

            target_summary_column_name = f"{data_type_label}_{session_label}_QC_summary"
            target_comment_column_name = f"{data_type_label}_{session_label}_QC_comment"

            # Ensure destination columns exist; create if not present
            for target_column_name in [target_summary_column_name, target_comment_column_name]:
                if target_column_name not in data_df.columns:
                    data_df[target_column_name] = np.nan
                    if target_column_name not in created_columns:
                        print(f"[info] Creating missing column in data_df: {target_column_name}")
                        created_columns.add(target_column_name)

            # Fetch QC values from QC_df row
            qc_summary_value = subject_rows_qc.at[row_index, "QC_summary"]
            qc_comment_value = subject_rows_qc.at[row_index, "QC_comment"]

            # Enforce: QC_summary must be present (error if missing/empty)
            if pd.isna(qc_summary_value) or (isinstance(qc_summary_value, str) and qc_summary_value.strip() == ""):
                raise ValueError(
                    f"[QC_df] Missing QC_summary for subject_ID={subject_identifier}, "
                    f"data_type={data_type_label}, session_ID={session_label}")

            # Write QC_summary (stringified)
            data_df.loc[data_df["subject_ID"].astype(str) == subject_identifier, target_summary_column_name] = str(qc_summary_value)

            # Write QC_comment if present; else ensure NaN
            if pd.isna(qc_comment_value) or (isinstance(qc_comment_value, str) and qc_comment_value.strip() == ""):
                data_df.loc[data_df["subject_ID"].astype(str) == subject_identifier, target_comment_column_name] = np.nan
            else:
                data_df.loc[data_df["subject_ID"].astype(str) == subject_identifier, target_comment_column_name] = str(qc_comment_value)

            updates_made += 1
            subjects_with_updates.add(subject_identifier)

    print(f"[result] QC transfer complete. Updated {updates_made} cell(s) across {len(subjects_with_updates)} subject(s).")

Next: fill 'first_good_<data_type>' columns:

- Note: updated to string-match for any strings provided in the 'QC_pass_labels' field of the config.yaml file (loaded in here as 'PASS_QC_LABELS' global var) -- this lets users set their own inclusion criteria without assuming our current labeling scheme.

In [ ]:
# =========================
# Compute first_good_MRI / first_good_MEG from *_QC_summary columns
# (earliest session with QC_summary ∈ PASS_QC_LABELS, ordered by TIME_POINTS_DICT keys)
# =========================

if not SKIP_CATALOGUE_QC:

    # 1) Chronological order of *base* sessions comes from dict key order
    ordered_base_sessions = list(TIME_POINTS_DICT.keys())
    base_session_to_rank = {label: rank for rank, label in enumerate(ordered_base_sessions)}

    def _split_base_and_run(session_label: str):
        """
        Accepts:
          - 'baseline'      -> ('baseline', 1)
          - 'baseline-2'    -> ('baseline', 2)
        Only treats a trailing '-<int>' as a run index.
        """
        session_label = str(session_label).strip()
        match = re.match(r"^(?P<base>.+?)-(?P<run>\d+)$", session_label)
        if match:
            return match.group("base"), int(match.group("run"))
        return session_label, 1

    def _session_sort_key(session_label: str):
        base, run = _split_base_and_run(session_label)
        if base not in base_session_to_rank:
            return (10**9, 10**9)  # push unknown bases to the end (and effectively exclude later)
        return (base_session_to_rank[base], run)

    # 2) Discover available *_QC_summary columns and parse dtype/session with regex
    qc_summary_pattern = re.compile(r"^(?P<dtype>[^_]+)_(?P<session>[^_]+)_QC_summary$")
    available_qc_summary_columns = [col for col in data_df.columns if col.endswith("_QC_summary")]

    parsed_summary_columns = []  # list of (column_name, dtype, session_full)
    available_data_types = set()

    for column_name in available_qc_summary_columns:
        match = qc_summary_pattern.match(column_name)
        if not match:
            continue

        data_type_label = match.group("dtype")
        session_full = match.group("session")

        base_session, run_index = _split_base_and_run(session_full)

        # Include any session whose *base* is in the ordered base-session keys
        if base_session in base_session_to_rank:
            parsed_summary_columns.append((column_name, data_type_label, session_full))
            available_data_types.add(data_type_label)

    # 3) Require MRI and MEG presence via *_QC_summary columns
    required_types = {"MRI", "MEG"}
    missing_types = sorted(list(required_types - available_data_types))
    if missing_types:
        raise ValueError(
            f"Missing *_QC_summary columns for required data types: {missing_types}. "
            f"Expected columns like '<type>_<session>_QC_summary'.")

    # 4) Ensure destination columns exist
    for destination_column in ["first_good_MRI", "first_good_MEG"]:
        if destination_column not in data_df.columns:
            data_df[destination_column] = np.nan

    # 5) Build per-type lists of (column_name, session) in chronological order
    columns_by_type = {dtype: [] for dtype in available_data_types}
    for column_name, data_type_label, session_label in parsed_summary_columns:
        columns_by_type[data_type_label].append((column_name, session_label))

    for data_type_label in columns_by_type:
        # sort columns by (base session rank; run index):
        columns_by_type[data_type_label].sort(key=lambda x: _session_sort_key(x[1]))

    # 6) For each required type, pick the earliest session with QC_summary ∈ PASS_QC_LABELS
    for data_type_label in ["MRI", "MEG"]:
        if data_type_label not in columns_by_type or not columns_by_type[data_type_label]:
            raise ValueError(f"No *_QC_summary columns found for data_type='{data_type_label}'.")

        destination_column = f"first_good_{data_type_label}"
        earliest_labels_per_row = []

        # iterate rows once for this dtype
        for row_index in range(len(data_df)):
            earliest_session_found = np.nan
            # scan sessions in chronological order
            for column_name, session_label in columns_by_type[data_type_label]:
                cell_value = data_df.at[row_index, column_name]
                if pd.isna(cell_value):
                    continue
                cell_text = str(cell_value).strip().lower()
                if cell_text in PASS_QC_LABELS:
                    earliest_session_found = session_label
                    break  # earliest found; stop scanning this row
            earliest_labels_per_row.append(earliest_session_found)

        data_df[destination_column] = earliest_labels_per_row

    # 7) Summaries
    for data_type_label in ["MRI", "MEG"]:
        destination_column = f"first_good_{data_type_label}"
        non_null_count = int(data_df[destination_column].notna().sum())
        value_counts = data_df[destination_column].value_counts(dropna=True).to_dict()
        print(f"[result] {destination_column}: set for {non_null_count} subject(s). Value counts → {value_counts}")

Next, we fill the 'n_good_MRI' and 'n_good_MEG' columns:

In [ ]:
# =========================
# Compute n_good_MRIs / n_good_MEGs from *_QC_summary columns
# (count how many sessions have QC_summary ∈ PASS_QC_LABELS per data_type)
# =========================

if not SKIP_CATALOGUE_QC:

    # 1) Discover all *_QC_summary columns and parse <data_type> / <session_ID>
    qc_summary_pattern = re.compile(r"^(?P<dtype>[^_]+)_(?P<session>[^_]+)_QC_summary$")
    qc_summary_columns = [column_name for column_name in data_df.columns if column_name.endswith("_QC_summary")]

    columns_by_type = {}  # { dtype: [col1, col2, ...] }
    for column_name in qc_summary_columns:
        match = qc_summary_pattern.match(column_name)
        if match:
            data_type_label = match.group("dtype")
            columns_by_type.setdefault(data_type_label, []).append(column_name)

    # 2) Ensure destination columns exist for MRI/MEG
    for destination_column in ["n_good_MRIs", "n_good_MEGs"]:
        if destination_column not in data_df.columns:
            data_df[destination_column] = np.nan

    # 3) For each dtype, count QC_summary ∈ PASS_QC_LABELS (case-sensitive; list used as-is)
    def count_pass_rows(subframe):
        # subframe: DataFrame with only the *_QC_summary columns for a given dtype
        # Trim whitespace but keep case; match exactly against PASS_QC_LABELS
        trimmed = subframe.applymap(lambda x: str(x).strip() if pd.notna(x) else x)
        is_pass = trimmed.applymap(lambda v: v in PASS_QC_LABELS)
        return is_pass.sum(axis=1).astype("Int64")  # nullable integer

    # MRI
    if "MRI" in columns_by_type and columns_by_type["MRI"]:
        mri_cols = columns_by_type["MRI"]
        data_df["n_good_MRIs"] = count_pass_rows(data_df[mri_cols])

        # Sanity check: counts should never exceed number of available columns
        max_possible_mri = len(mri_cols)
        too_high_mask = data_df["n_good_MRIs"] > max_possible_mri
        if too_high_mask.any():
            rows_bad = too_high_mask.sum()
            raise ValueError(f"[sanity] n_good_MRIs exceeds max possible ({max_possible_mri}) for {rows_bad} row(s).")
    else:
        print("[info] No MRI *_QC_summary columns found; leaving 'n_good_MRIs' as NaN.")

    # MEG
    if "MEG" in columns_by_type and columns_by_type["MEG"]:
        MEG_cols = columns_by_type["MEG"]
        data_df["n_good_MEGs"] = count_pass_rows(data_df[MEG_cols])

        # Sanity check
        max_possible_MEG = len(MEG_cols)
        too_high_mask = data_df["n_good_MEGs"] > max_possible_MEG
        if too_high_mask.any():
            rows_bad = too_high_mask.sum()
            raise ValueError(f"[sanity] n_good_MEGs exceeds max possible ({max_possible_MEG}) for {rows_bad} row(s).")
    else:
        print("[info] No MEG *_QC_summary columns found; leaving 'n_good_MEGs' as NaN.")

    # 4) Summary
    if "n_good_MRIs" in data_df:
        print(f"[result] n_good_MRIs  — non-null rows: {int(data_df['n_good_MRIs'].notna().sum())}, "
              f"min/max: {data_df['n_good_MRIs'].min()} / {data_df['n_good_MRIs'].max()}")
    if "n_good_MEGs" in data_df:
        print(f"[result] n_good_MEGs — non-null rows: {int(data_df['n_good_MEGs'].notna().sum())}, "
              f"min/max: {data_df['n_good_MEGs'].min()} / {data_df['n_good_MEGs'].max()}")

If we are SKIPPING QC decoration, set columns to show 'unknown':

In [ ]:
if not SKIP_CATALOGUE_QC:
    pass
else:
    data_df = data_df.drop(columns=[c for c in data_df.columns if '_QC_comment' in c], errors='ignore')
    unknown_columns = ['n_good_MRIs', 'n_good_MEGs', 'first_good_MRI', 'first_good_MEG'] + [c for c in data_df.columns if '_QC_summary' in c]
    data_df[unknown_columns] = data_df[unknown_columns].fillna('[unknown]')

data_df.head(7)

-----------
#### Final save / export:

In [ ]:
output_file_path = Path(ROOT_DIRECTORY) / "master_data_catalogue.csv"

# Export the dataframe to .csv:
data_df.to_csv(output_file_path, index=False)